In [2]:
import pandas as pd
import numpy as np
from src.data import download_labr, load_reviews, load_split_ids, normalize, DIAC

import sys
from pathlib import Path
ROOT = Path.cwd().parent
sys.path.append(str(ROOT))
DATA = ROOT / "data"

pd.set_option("display.max_colwidth", 120)

download_labr(DATA)

already here: reviews.tsv
already here: 5class-unbalanced-train.txt
already here: 5class-unbalanced-test.txt


In [3]:
df = load_reviews(DATA)
tr_ids, te_ids = load_split_ids(DATA)
print("rows:", len(df), "| train:", len(tr_ids), "| test:", len(te_ids))
print("train∩test overlap:", len(set(tr_ids) & set(te_ids)))
print("ids cover all rows:", len(set(tr_ids) | set(te_ids)) == len(df))
df.head(3)

rows: 63257 | train: 50606 | test: 12651
train∩test overlap: 0
ids cover all rows: True


,rating,review_id,user_id,book_id,text
0,4,338670838,7878381,13431841,"""عزازيل الذي صنعناه ،الكامن في أنفسنا"" يذكرني يوسف زيدان بــ بورخس في استخدامه لحيلته الفنية،وخداع القاريء بأن الرو..."
1,4,39428407,1775679,3554772,من أمتع ما قرأت من روايات بلا شك. وحول الشك تدندن (عزازيل) بلا هوادة. أحمد الديب 2008
2,4,32159373,1304410,3554772,رواية تتخذ من التاريخ ،جوًا لها اختار المؤلف فترة تاريخية ندر من يتناولها روائيًا. مكتوبة بدقة وإتقان وجمال.من أروع...


In [4]:
df["split"] = "train"
# te_ids are positions -> df.index[...] converts them to row labels
df.loc[df.index[te_ids], "split"] = "test"
dist = pd.crosstab(df.rating, df.split, normalize="columns").round(
    3)   # proportions within each split
dist["count_all"] = df.rating.value_counts().sort_index()
dist

split,test,train,count_all
rating,,,
1,0.048,0.046,2939
2,0.086,0.083,5285
3,0.187,0.194,12201
4,0.303,0.301,19054
5,0.376,0.376,23778


In [5]:
df["n_words"] = df.text.str.split().str.len()
print(df.n_words.describe().round(1))
print("empty:", (df.n_words == 0).sum(),
      "| < 3 words:", (df.n_words < 3).sum())
# median, not mean: a few huge reviews would distort the mean
df.groupby("rating").n_words.median()

count    63257.0
mean        62.0
std        108.1
min          1.0
25%         14.0
50%         31.0
75%         67.0
max       3419.0
Name: n_words, dtype: float64
empty: 0 | < 3 words: 1548


rating
1    26.0
2    33.0
3    35.0
4    34.0
5    28.0
Name: n_words, dtype: float64

In [6]:
print("with diacritics:", df.text.str.contains(DIAC).sum())
print("with tatweel   :", df.text.str.contains("\u0640").sum())
print("with elongation:", df.text.str.contains(r"(.)\1{2,}").sum())
print("with latin     :", df.text.str.contains(r"[A-Za-z]").sum())
print("with '!'       :", df.text.str.contains("!").sum())
df["norm"] = df.text.map(normalize)
df.loc[df.text.str.contains("\u0640"), ["text", "norm"]].head(
    3)   # before/after examples

with diacritics: 19674
with tatweel   : 3197


/tmp/ipykernel_307483/3768188284.py:3: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  print("with elongation:", df.text.str.contains(r"(.)\1{2,}").sum())


with elongation: 6146
with latin     : 0
with '!'       : 11792


,text,norm
0,"""عزازيل الذي صنعناه ،الكامن في أنفسنا"" يذكرني يوسف زيدان بــ بورخس في استخدامه لحيلته الفنية،وخداع القاريء بأن الرو...","""عزازيل الذي صنعناه ،الكامن في انفسنا"" يذكرني يوسف زيدان ب بورخس في استخدامه لحيلته الفنيه،وخداع القاريء بان الروايه..."
12,"ماذا أقول فى رواية خطفت عقلى خطفا, ثم سلمتنى عقلا آخر جديدا؟ ماذا أقول فى رواية يستنبط عقلى من كل كلمة منها فكرة غا...","ماذا اقول في روايه خطفت عقلي خطفا, ثم سلمتني عقلا اخر جديدا؟ ماذا اقول في روايه يستنبط عقلي من كل كلمه منها فكره غائ..."
24,عزازيل، إبليس، الشيطان، أهريمان، بعلزبوب، بعلزبول. لا تستغرب فأنا أعي ما أكتب، هذه كلها أسماء تدل على إبليس، وتدل ف...,عزازيل، ابليس، الشيطان، اهريمان، بعلزبوب، بعلزبول. لا تستغرب فانا اعي ما اكتب، هذه كلها اسماء تدل علي ابليس، وتدل في...


In [7]:
print("exact dup texts     :", df.text.duplicated().sum())
# always >= exact: normalization reveals hidden copies
print("normalized dup texts:", df.norm.duplicated().sum())
# all empty reviews are "duplicates" of each other; not meaningful
nonempty = df[df.norm != ""]
# number of distinct ratings per unique text
conf = nonempty.groupby("norm").rating.nunique()
print("contradictory groups:", (conf > 1).sum())
# .iloc = by position, matching how the id files work
tr_set = set(df.iloc[tr_ids].norm)
te_norm = df.iloc[te_ids].norm
print("test texts also in train (leak):",
      (te_norm.isin(tr_set) & (te_norm != "")).sum())
nonempty[nonempty.norm.isin(conf[conf > 1].index)].sort_values("norm")[
    ["rating", "norm"]].head(8)

exact dup texts     : 3167
normalized dup texts: 3342
contradictory groups: 140
test texts also in train (leak): 988


,rating,norm
1056,5,""" اكتب يا هيبا ، فمن يكتب لن يموت ابدا"""
807,4,""" اكتب يا هيبا ، فمن يكتب لن يموت ابدا"""
9271,5,ا
35059,4,ا
6170,5,اتشرف بدعوه حضراتكم لزياره صفحه اول مكتبه علي فيسبوك لبيع الكتب بسعر الجمله والتوصيل مجاني للاستفسار : 01203374646
12528,3,اتشرف بدعوه حضراتكم لزياره صفحه اول مكتبه علي فيسبوك لبيع الكتب بسعر الجمله والتوصيل مجاني للاستفسار : 01203374646
6471,5,اتشرف بدعوه حضراتكم لزياره صفحه اول مكتبه علي فيسبوك لبيع الكتب بسعر الجمله والتوصيل مجاني للاستفسار : 01203374646
41012,5,اتشرف بدعوه حضراتكم لزياره صفحه اول مكتبه علي فيسبوك لبيع الكتب بسعر الجمله والتوصيل مجاني للاستفسار : 01203374646


In [8]:
# -1 so that 0-word reviews fall in the first bucket
bins = [-1, 5, 20, 50, 100, 100_000]
df.groupby(pd.cut(df.n_words, bins)).rating.agg(["mean", "count"]).round(2)

/tmp/ipykernel_307483/457250689.py:3: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby(pd.cut(df.n_words, bins)).rating.agg(["mean", "count"]).round(2)


,mean,count
n_words,,
"(-1, 5]",3.99,5351
"(5, 20]",3.91,17509
"(20, 50]",3.86,18850
"(50, 100]",3.83,11748
"(100, 100000]",3.84,9799


In [9]:
import sys
sys.path.append("..")
from src.data import make_splits, save_splits


train, val, test, log = make_splits(df, tr_ids, te_ids)
print(log)

# proof of no leakage: number of normalized texts shared between each pair of splits (all must be 0)
print("train∩val:", len(set(train.norm) & set(val.norm)),
      "| train∩test:", len(set(train.norm) & set(test.norm)),
      "| val∩test:", len(set(val.norm) & set(test.norm)))

# proof of stratification: rating proportions per split
print(pd.concat({k: v.rating.value_counts(normalize=True).sort_index().round(3)
                 for k, v in {"train": train, "val": val, "test": test}.items()}, axis=1))

save_splits({"train": train, "val": val, "test": test}, DATA / "splits")

{'train_official': 50606, 'removed_leak': 1352, 'removed_dups_or_ties': 1830, 'train': 42681, 'val': 4743, 'test': 12651}
train∩val: 0 | train∩test: 0 | val∩test: 0
        train    val   test
rating                     
1       0.047  0.047  0.048
2       0.084  0.084  0.086
3       0.196  0.196  0.187
4       0.302  0.301  0.303
5       0.372  0.372  0.376
